In [0]:
# Run common utility functions and variables from the helpers notebook
%run ../helpers/common_utilities

In [0]:
# Get the current user's username from the notebook context
user_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

# Define the pipeline name for IOT sensor data ingestion
pipeline_name = "IOT_sensor_data_ingest"

In [0]:
# Create catalog if it does not exist for organizing IOT data assets
query = f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}"
print(query)
spark.sql(query)

# Create volume if it does not exist, used for storing IOT sensor data files
query = f"""CREATE VOLUME IF NOT EXISTS {CATALOG_VOLUME}
COMMENT 'Volume for storing IOT sensor data' """
print(query)
spark.sql(query)

In [0]:
# Define the volume location for the database's external storage
volume_location = f"/Volumes/{CATALOG_NAME}/{Default_schema}/{Volume_name}"

# Create the database for IOT raw sensor data if it does not exist
query = f"""
CREATE DATABASE IF NOT EXISTS {BRONZE_DATABASE}
COMMENT 'Database for IOT raw sensor data'
"""

print(query)
# Execute the SQL statement to create the database
spark.sql(query)

In [0]:
# Define the pipeline configuration payload for Databricks pipeline creation
pipeline_payload = {
    "name": pipeline_name,  # Name of the pipeline
    "catalog": CATALOG_NAME,  # Catalog where pipeline assets are stored
    "target": Bronze_schema,  # Target schema for pipeline output
    "libraries": [
        {
            "notebook": {
                # Path to the notebook that processes raw sensor data events
                "path": f"/Workspace/Repos/{user_name}/heavy-electrical-predictive-maintenance/Bronze_iot_creation/Raw_sensor_data_events"
            }
        }
    ],
    "channel": "current",  # Use the current channel for pipeline execution
    "serverless": True     # Enable serverless compute for the pipeline
}
print(pipeline_payload)

# Initialize the pipeline manager and create the pipeline if it does not exist
pipeline_manager = DatabricksPipelineManager(pipeline_name, pipeline_payload)
pipeline_manager.create_pipeline_if_not_exists()